In [0]:
dbutils.widgets.text("run_date", "", "Run Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, coalesce, when

spark = SparkSession.builder.getOrCreate()

env = aws_utils.get_env_prefix()

# If run_date not provided, use latest available BusinessDate from exposure
run_date_param = dbutils.widgets.get("run_date").strip()

if run_date_param:
    run_date = run_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")

print(f"env:       {env}")
print(f"run_date:  {run_date}")

# dtransfer_Res

Replicates `[PLTFRM].[dtransfer].[ResSp]` — real estate properties with ownership, excluding collateral.

## Sources
- `drealestate` (thesis only) — property details, categories, location, valuation
- `realestateinfo` — ownership (PartyId, OwnershipPercentage, OwnershipType, PropertyOwnershipId)
- `dtransfer_collateralre` — anti-join exclusion (mirrors on-prem NOT EXISTS)

## Key Decisions
- **thesis only, Active only** — QCA rows carry raw codes; `RealEstateActive='Active'` aligns with on-prem `Property.active=1`
- **Hybrid ownership filter** — keep Active/NULL + Inactive rows whose property has ≥1 Active sibling; aligns with on-prem `StatusRecord=1` (99.8% recall vs 90.2% with pure `!= 'Inactive'`)
- **Address** = `TRIM(RealEstateAddress1)`, not `Address` (which concats Addr1+Addr2)
- **LastValuationDate** = `COALESCE(RealEstateValueDate, RealEstateEvaluationDate, '1900-01-01')` — confirmed ceiling vs on-prem
- **PartyId** = `ri.OwnerPartyKey` (cloud-native party key); on-prem uses `PropertyOwnership.Personid` (DWH PersonId) — different ID system, 0% match by design (same pattern as LoanId in platformEOM)
- **Dependency**: `dtransfer_collateralre` must run first

## Output
- `{env}-cdh-delivery.it.dtransfer_res` — daily overwrite

In [0]:
from pyspark.sql.functions import trim

#Sources 
# thesis only
drealestate = (
    spark.table(f"`{env}-cdh`.derivative.drealestate")
    .filter((col("BusinessDate") == run_date_lit) & (col("SourceSystem") == "thesis") & (col("RealEstateActive") == "Active"))  # Property.active=1
)


_cre = spark.table(f"`{env}-cdh-delivery`.it.dtransfer_collateralre")
_cre_max_date = _cre.agg({"ReferenceDate": "max"}).collect()[0][0]
print(f"collateralre date: {_cre_max_date} (run_date: {run_date})")
if str(_cre_max_date) != run_date:
    print(f"  NOTE: date mismatch — run CollateralRE notebook for {run_date} to align")

dcollateral_keys = (
    _cre
    .filter(col("ReferenceDate") == lit(str(_cre_max_date)).cast("date"))
    .select(col("PropertyKey").cast("string").alias("ExcludeKey")) 
    .distinct()
)

# Ownership
_ri_all = spark.table(f"`{env}-cdh`.primaryactive.realestateinfo").filter(col("BusinessDate") == run_date_lit)
_active_rekeys = _ri_all.filter(col("OwnershipStatusCode") != "Inactive").select(col("RealEstateKey").alias("ak_key")).distinct()

realestateinfo = (
    _ri_all.filter(col("OwnershipStatusCode") != "Inactive")
    .unionByName(
        _ri_all.filter(col("OwnershipStatusCode") == "Inactive")
        .join(_active_rekeys, col("RealEstateKey") == col("ak_key"), "inner")
        .drop("ak_key")
    )
)

# Build Res 
res_df = (
    drealestate.alias("re")
    .join(dcollateral_keys, col("re.RealEstateKey") == dcollateral_keys["ExcludeKey"], "left_anti")
    .join(realestateinfo.alias("ri"), col("re.RealEstateKey") == col("ri.RealEstateKey"), "inner")
    .select(
        lit(run_date).cast("timestamp").alias("EtlLoadingDatetime"),
        run_date_lit.alias("ReferenceDate"),
        col("re.RealEstateCepalKey").alias("PropertyKey"),
        col("re.RealEstateCategory").alias("ParentCategory"),
        col("re.RealEstateSubCategory").alias("PropertyCategory"),
        col("re.RealEstateDescription").alias("PropertyDescription"),
        col("re.RealEstateMarketValue").alias("MarketValue"),
        coalesce(col("re.RealEstateValueDate"), col("re.RealEstateEvaluationDate"),
                 lit("1900-01-01").cast("date")).alias("LastValuationDate"),
        col("re.RealEstateCity").alias("City"),
        col("re.RealEstateArea").alias("Area"),
        col("re.RealEstatePostalCode").alias("ZipCode"),
        trim(col("re.RealEstateAddress1")).alias("Address"),  
        coalesce(col("re.RealEstateDistrict"), lit("N/A")).alias("Prefecture"),
        col("ri.OwnershipKey").alias("PropertyOwnershipId"),
        col("ri.OwnershipPercentage"),
        col("ri.OwnershipType"),
        col("ri.OwnerPartyKey").alias("PartyId"),
        col("re.RealEstateUsageCodeTypeCode").alias("UsageCode"),
    )
    .distinct()
)

print(f"dtransfer_res rows: {res_df.count():,}")

In [0]:
target_res = f"`{env}-cdh-delivery`.it.dtransfer_res"

(
    res_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_res)
)

print(f"Written to {target_res}")